# Syria Population Distribution Map (2026)

Overview

Visualises the estimated population distribution across Syria using the WorldPop 2026 dataset. Estimated population values are displayed as a raster overlay on a dark basemap with administrative boundaries and governorate labels.

WorldPop 2026を用いて、シリアにおける推計人口分布を可視化します。推計人口値を、行政界および県名を重ねた暗色ベースマップ上にラスタとして表示します。

Objectives

- Read and process the WorldPop population raster
- Downsample the raster for web-map display
- Apply a custom colour map
- Display the population raster with administrative boundaries
- Add a colour legend and information panel
- Export the completed map as an HTML file

- WorldPopの人口ラスタを読み込んで処理する
- Web地図表示用にラスタを縮小する
- 独自のカラーマップを適用する
- 行政界とともに人口ラスタを表示する
- カラー凡例と情報パネルを追加する
- 完成した地図をHTMLファイルとして保存する

Workflow

1. Define the input and output paths
2. Read and downsample the population raster
3. Preserve NoData areas as transparent pixels
4. Create and normalise the custom colour map
5. Create a dark basemap without built-in place labels
6. Add administrative boundaries and place labels
7. Add the population raster, legend and information panel
8. Save and display the interactive map

Data

Population raster data:

- `worldpop_syria_2026.tif`

Source: WorldPop, 2026 estimated population dataset

Administrative boundary data:

- `syr_admin0.geojson`
- `syr_admin1.geojson`

Source: HDX OCHA, Syria subnational administrative boundaries

Data Scope and Limitations

- Population values are estimates rather than census counts.
- The source raster resolution is 3 arc seconds, approximately 100 metres.
- The raster is downsampled using average resampling for web-map display.
- The downsampled display layer should not be used to calculate population totals.
- Values above 100 are displayed using the highest colour class.
- The 0–100 display range changes only the map colours and does not modify the source raster.

- 人口値は国勢調査による実測値ではなく推計値です。
- 元ラスタの解像度は3秒角で、約100メートルです。
- Web地図表示用のラスタには平均値による縮小処理を行っています。
- 縮小後の表示レイヤーは、人口総数の計算には使用できません。
- 100を超える値は、最上位の色で表示します。
- 0〜100の表示範囲は地図上の色だけに適用され、元ラスタの値は変更しません。

Technologies

- Python
- Rasterio
- Folium
- Matplotlib
- Branca
- pathlib

In [ ]:
# 1
# Import the required libraries
# 必要なライブラリを読み込む

from pathlib import Path

import branca
import folium
import matplotlib.colors as mcolors
import rasterio
from rasterio.enums import Resampling

In [ ]:
# 2
# Define the input and output paths
# 入力データと出力ファイルのパスを設定する

PROJECT_DIR = Path.cwd()
ROOT_DIR = PROJECT_DIR.parents[1]

VECTOR_DIR = ROOT_DIR / "02_DATA" / "VECTOR"
RASTER_DIR = ROOT_DIR / "02_DATA" / "RASTER"

admin0_path = VECTOR_DIR / "syr_admin0.geojson"
admin1_path = VECTOR_DIR / "syr_admin1.geojson"

population_path = RASTER_DIR / "worldpop_syria_2026.tif"

output_path = PROJECT_DIR / "01_syria_population_distribution_map.html"

In [ ]:
# 3
# Read and downsample the population raster
# 人口ラスタを読み込み、表示用に縮小する

with rasterio.open(population_path) as src:

    # Reduce the raster dimensions by half.
    # 縦横を半分にし、Web表示用のサイズへ縮小する

    display_height = (src.height + 1) // 2
    display_width = (src.width + 1) // 2

    # Use average resampling instead of discarding every second pixel.
    # 1つおきに画素を捨てず、周辺画素の平均値を利用する

    pop_array = src.read(
        1,
        out_shape=(display_height, display_width),
        masked=True,
        resampling=Resampling.average,
    )

    # Convert NoData pixels to NaN so that they can be transparent.
    # NoData領域を透明表示できるよう、NaNへ変換する

    pop_array = pop_array.filled(float("nan"))

    # Extract raster bounds for Folium.
    # Foliumで使用するラスタの表示範囲を取得する

    bounds = [[src.bounds.bottom, src.bounds.left], [src.bounds.top, src.bounds.right]]

In [ ]:
# 4
# Create and normalise the custom colour map
# オリジナルのカラーマップを作成し、人口値を正規化する

# Define the custom colour palette
# 独自のカラーパレットを定義する

colors = [
    (0.00, "#00176100"),
    (0.20, "#297b8eff"),
    (0.40, "#28ae80ff"),
    (0.60, "#2eb37cff"),
    (0.80, "#e5e419ff"),
    (1.00, "#fde725ff"),
]

marisa_cmap = mcolors.LinearSegmentedColormap.from_list("Syria_Neon", colors)

# Make missing raster values transparent.
# 欠損値を透明にする

marisa_cmap.set_bad((0, 0, 0, 0))

# Display population values from 0 to 100.
# 100を超える値は最高色として表示する

POPULATION_MIN = 0.0
POPULATION_MAX = 100.0

population_norm = mcolors.Normalize(vmin=POPULATION_MIN, vmax=POPULATION_MAX, clip=True)


def population_colormap(value):
    """
    Convert a WorldPop value to an RGBA colour.
    WorldPopの人口値をRGBAカラーへ変換する。
    """

    # NaN values represent NoData pixels.
    # NaNはNoDataを表すため、透明にする

    if value is None or value != value:
        return (0, 0, 0, 0)

    return marisa_cmap(population_norm(value))

In [ ]:
# 5
# Create a dark basemap without built-in labels
# 地名表記のない暗色ベースマップを作成する

m = folium.Map(
    location=[34.8, 38.5],
    zoom_start=7,
    tiles=("https://{s}.basemaps.cartocdn.com/" "dark_nolabels/{z}/{x}/{y}{r}.png"),
    attr=(
        '&copy; <a href="https://www.openstreetmap.org/copyright">'
        "OpenStreetMap</a> contributors "
        '&copy; <a href="https://carto.com/attributions">CARTO</a>'
    ),
)

In [ ]:
# 6
# Add the country and governorate boundaries
# 国境および県境を追加する

folium.GeoJson(
    str(admin0_path),
    name="Country",
    style_function=lambda feature: {"color": "white", "weight": 3, "fillOpacity": 0},
).add_to(m)

folium.GeoJson(
    str(admin1_path),
    name="Governorate",
    style_function=lambda feature: {"color": "gray", "weight": 1, "fillOpacity": 0},
).add_to(m)

In [ ]:
# 7
# Add neighbouring-country labels
# 周辺国名を追加する

neighbors = {
    "TÜRKIYE": [37.5, 37.5],
    "IRAQ": [34.5, 42.0],
    "JORDAN": [31.9, 36.5],
    "LEBANON": [34.2, 35.0],
}

for name, coords in neighbors.items():
    folium.Marker(
        location=coords,
        icon=folium.DivIcon(
            html=f"""
            <div style="
                font-size: 14pt;
                font-weight: bold;
                color: lightgray;
                white-space: nowrap;
                text-align: center;
                width: 100px;
                margin-left: -50px;
            ">
                {name}
            </div>
            """
        ),
    ).add_to(m)

In [ ]:
# 8
# Add governorate labels
# シリア14県の名称を追加する

governorates = {
    "Aleppo": [36.2, 37.5],
    "Al-Hasakeh": [36.5, 40.7],
    "Ar-Raqqa": [36.0, 39.0],
    "As-Sweida": [32.8, 36.9],
    "Daraa": [32.9, 36.2],
    "Deir-ez-Zor": [35.1, 40.5],
    "Damascus": [33.7, 36.7],
    "Hama": [35.2, 37.0],
    "Homs": [34.5, 38.3],
    "Idleb": [35.8, 36.7],
    "Lattakia": [35.6, 36.1],
    "Quneitra": [33.1, 35.9],
    "Rural Damascus": [33.5, 37.5],
    "Tartous": [34.9, 36.1],
}

for name, coords in governorates.items():
    folium.Marker(
        location=coords,
        icon=folium.DivIcon(
            html=f"""
            <div style="
                font-size: 10pt;
                color: white;
                font-weight: bold;
                white-space: nowrap;
                text-align: center;
                width: 100px;
                margin-left: -50px;
            ">
                {name}
            </div>
            """
        ),
    ).add_to(m)

In [ ]:
# 9
# Add the population raster overlay
# 人口ラスタを地図へ重ねる

population_overlay = folium.raster_layers.ImageOverlay(
    image=pop_array,
    bounds=bounds,
    colormap=population_colormap,
    opacity=0.8,
    name="Estimated Population",
)

population_overlay.add_to(m)

In [ ]:
# 10
# Add and style the population colour legend
# 人口値のカラー凡例を追加し、表示を整える

legend_values = [0, 20, 40, 60, 80, 100]

legend_colors = [marisa_cmap(population_norm(value)) for value in legend_values]

colormap = branca.colormap.LinearColormap(
    colors=legend_colors,
    index=legend_values,
    vmin=POPULATION_MIN,
    vmax=POPULATION_MAX,
    caption=("Estimated population per source " "100 m grid cell"),
)

m.add_child(colormap)


# Style the legend for the dark basemap
# ダーク地図に合わせてカラーバーの外観を整える

dark_legend_css = """
<style>
    .legend {
        background: rgba(0, 0, 0, 0.7) !important;
        color: white !important;
        padding: 12px !important;
        border: 1px solid rgba(255, 255, 255, 0.4) !important;
        border-radius: 8px !important;
    }

    .legend text {
        fill: white !important;
        font-size: 12px !important;
        font-weight: bold !important;
    }

    .legend caption {
        color: white !important;
        font-weight: bold !important;
    }
</style>
"""

m.get_root().header.add_child(folium.Element(dark_legend_css))

In [ ]:
# 11
# Add the map information and source panel
# 地図の説明と出典を表示するパネルを追加する

information_panel_html = """
<div style="
    position: fixed;
    top: 20px;
    left: 50px;
    width: 360px;
    min-height: 150px;
    background-color: rgba(0, 0, 0, 0.75);
    color: white;
    z-index: 9000;
    font-size: 14px;
    border: 1px solid white;
    border-radius: 8px;
    padding: 12px;
    box-shadow: 0 0 15px rgba(0, 0, 0, 0.5);
">
    <b style="font-size: 16px;">
        Syria
    </b>
    <br>

    <span style="color: #ffd700;">
        Population Distribution (2026 Estimated)
    </span>
    <br>

    <small style="
    display: block;
    margin-top: 5px;
    line-height: 1.2;
    color: #eee;
">
    Values represent estimated population per source
    grid cell. The source resolution is 3 arc seconds
    (approximately 100 m). Average resampling is used
    to create the display layer.
</small>

    <div style="
        margin-top: 10px;
        padding-top: 5px;
        border-top: 1px solid #444;
        font-size: 11px;
        color: #bbb;
    ">
        Source:
        <a
            href="https://hub.worldpop.org/geodata/summary?id=75632"
            target="_blank"
            style="color: #3498db; text-decoration: none;"
        >
            WorldPop (Open Access Data)
        </a>
    </div>
</div>
"""

m.get_root().html.add_child(folium.Element(information_panel_html))

In [ ]:
# 12
# Save and display the interactive map
# インタラクティブ地図を保存し、Notebook上に表示する

m.save(output_path)

print(f"Map saved to: {output_path}")

m